# Day 52 — Prompt Engineering Techniques
### 12+ patterns: zero-shot, few-shot, chain-of-thought, system prompts, structured output, tool use

## 1. Setup — Client & Helper Function

In [1]:
import os
import anthropic
import json

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

def ask(prompt, system=None, max_tokens=300):
    kwargs = {"model": MODEL, "max_tokens": max_tokens,
              "messages": [{"role": "user", "content": prompt}]}
    if system:
        kwargs["system"] = system
    response = client.messages.create(**kwargs)
    return response.content[0].text

In [2]:
import sys
print(sys.executable)

c:\DS-AI-75d\.venv\Scripts\python.exe


## 2. Pattern 1 — Zero-Shot Prompting

In [2]:
prompt = "Classify the sentiment of this review as positive, negative, or neutral: 'The food was cold but the service was excellent.'"

result = ask(prompt)
print(result)

# Sentiment Classification: **Mixed/Neutral**

This review contains **both positive and negative elements**:

- **Negative**: "The food was cold" (a significant complaint)
- **Positive**: "the service was excellent" (strong praise)

If forced to choose a single category, I'd classify it as **Neutral** since the praise and criticism balance each other out. However, it's more accurate to note this is a **mixed sentiment** review where the customer had a problematic experience (cold food) offset by good service recovery.


## 3. Pattern 2 — Few-Shot Prompting

In [3]:
prompt = """Classify sentiment as positive, negative, or neutral. Respond with one word only.

Review: "Absolutely loved it, best meal all year!"
Sentiment: positive

Review: "Waited an hour and the order was wrong."
Sentiment: negative

Review: "The food was cold but the service was excellent."
Sentiment:"""

result = ask(prompt)
print(result)

neutral


## 4. Pattern 3 — Chain-of-Thought (CoT) Prompting

In [4]:
prompt = """A store has 23 apples. They sell 8 in the morning and receive a delivery
of 15 more in the afternoon, then sell 12 before closing. How many apples are left?

Think through this step by step, then give the final answer."""

result = ask(prompt)
print(result)

# Step-by-Step Calculation

**Starting amount:** 23 apples

**Morning sale:** 23 - 8 = 15 apples

**Afternoon delivery:** 15 + 15 = 30 apples

**Evening sale:** 30 - 12 = 18 apples

# Final Answer

**18 apples** are left at the end of the day.


## 5. Pattern 4 — System Prompts (Role & Behaviour Control)

In [5]:
system_prompt = "You are a terse senior data engineer. You answer in maximum 2 sentences, no fluff, no pleasantries."

prompt = "What's the difference between a list and a tuple in Python?"

result = ask(prompt, system=system_prompt)
print(result)

Lists are mutable (can be modified after creation) and use square brackets; tuples are immutable (fixed once created) and use parentheses. Use tuples for fixed data or dict keys, lists for dynamic collections.


## 6. Pattern 5 — Structured JSON Output

In [6]:
prompt = """Extract the following details from this text and return ONLY valid JSON,
no explanation, no markdown formatting:

Text: "John Smith, 34, works as a software engineer at TechCorp in Seattle. He can be
reached at john.smith@email.com."

Return JSON with keys: name, age, job_title, company, city, email"""

result = ask(prompt)
print(result)

```json
{
  "name": "John Smith",
  "age": 34,
  "job_title": "software engineer",
  "company": "TechCorp",
  "city": "Seattle",
  "email": "john.smith@email.com"
}
```


## 7. Pattern 6 — Parsing JSON Output Reliably (Real-World Fix)

In [7]:
def extract_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    return json.loads(text.strip())

parsed = extract_json(result)
print(parsed)
print(type(parsed))
print(parsed['email'])

{'name': 'John Smith', 'age': 34, 'job_title': 'software engineer', 'company': 'TechCorp', 'city': 'Seattle', 'email': 'john.smith@email.com'}
<class 'dict'>
john.smith@email.com


## 8. Pattern 7 — Tool Use / Function Calling

In [8]:
tools = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a given city",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "The city name"}
            },
            "required": ["city"]
        }
    }
]

response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    tools=tools,
    messages=[{"role": "user", "content": "What's the weather like in Mumbai right now?"}]
)

for block in response.content:
    print(block.type)
    if block.type == "tool_use":
        print("Tool:", block.name)
        print("Input:", block.input)

tool_use
Tool: get_weather
Input: {'city': 'Mumbai'}


## 9. Pattern 8 — Role/Persona Prompting

In [9]:
prompt_as_expert = ask("Should I use a list or a generator for processing a 10GB file in Python?",
                        system="You are a senior Python performance engineer who always explains the memory implications of any choice.")

prompt_as_beginner = ask("Should I use a list or a generator for processing a 10GB file in Python?",
                          system="You are a friendly coding tutor explaining to someone who wrote their first Python script last week.")

print("EXPERT VERSION:\n", prompt_as_expert)
print("\n\nBEGINNER VERSION:\n", prompt_as_beginner)

EXPERT VERSION:
 # Use a **generator** – here's why:

## Memory Implications

| Approach | Memory Usage | Best For |
|----------|--------------|----------|
| **List** | ~10GB (entire file in RAM) | Small files, multiple passes |
| **Generator** | ~1-2MB (buffer only) | Large files, single pass |

## Practical Example

```python
# ❌ BAD: List approach
def read_file_list(filepath):
    lines = []
    with open(filepath) as f:
        for line in f:
            lines.append(line.strip())
    return lines

# Process 10GB file
all_lines = read_file_list('huge_file.txt')  # BLOCKS until entire file loaded
for line in all_lines:
    process(line)
```

**Memory impact:** The entire 10GB sits in RAM simultaneously. Your program will likely crash or freeze.

```python
# ✅ GOOD: Generator approach
def read_file_generator(filepath):
    with open(filepath) as f:
        for line in f:
            yield line.strip()

# Process 10GB file
for line in read_file_generator('huge_file.txt'):
    process(

## 10. Pattern 9 — Output Format Constraints

In [10]:
prompt = """List 5 Python libraries for data visualization.
Format: a numbered list, library name only, no descriptions, no extra text."""

result = ask(prompt)
print(result)
print("\n---")
print("Word count:", len(result.split()))

1. Matplotlib
2. Seaborn
3. Plotly
4. Bokeh
5. Altair

---
Word count: 10


## 11. Pattern 10 — XML Tags for Structuring Input

In [11]:
prompt = """<task>Summarize the customer feedback below in exactly one sentence.</task>

<feedback>
The checkout process took way too long, almost 5 minutes, and I nearly abandoned my cart.
But once I got through it, the product quality was excellent and shipping was fast.
</feedback>

<output_format>One sentence only, mention both the negative and positive point.</output_format>"""

result = ask(prompt)
print(result)

Although the lengthy checkout process nearly caused cart abandonment, the excellent product quality and fast shipping made the overall experience worthwhile.


## 12. Pattern 11 — Negative Examples (Explicit Do's and Don'ts)

In [12]:
prompt = """Write a product description for a wireless mouse.

Do:
- Keep it under 30 words
- Focus on practical benefits

Don't:
- Use superlatives like "best" or "amazing"
- Use exclamation marks
- Mention price"""

result = ask(prompt)
print(result)

# Wireless Mouse

Navigate your desk freely without tangled cables. Responsive tracking works on most surfaces. Long battery life means fewer interruptions. Comfortable grip reduces hand fatigue during extended use.


## 13. Pattern 12 — Temperature Comparison (Determinism vs Creativity)

In [13]:
prompt = "Write a one-line tagline for a coffee shop."

print("Temperature 0.0 (deterministic) — run 3 times:")
for i in range(3):
    response = client.messages.create(model=MODEL, max_tokens=50, temperature=0.0,
                                       messages=[{"role": "user", "content": prompt}])
    print(f"  {i+1}.", response.content[0].text)

print("\nTemperature 1.0 (creative) — run 3 times:")
for i in range(3):
    response = client.messages.create(model=MODEL, max_tokens=50, temperature=1.0,
                                       messages=[{"role": "user", "content": prompt}])
    print(f"  {i+1}.", response.content[0].text)

Temperature 0.0 (deterministic) — run 3 times:
  1. "Wake up to something worth savoring."
  2. "Wake up to something extraordinary."
  3. "Wake up to something extraordinary."

Temperature 1.0 (creative) — run 3 times:
  1. "Brew the day right, one cup at a time."
  2. "Wake up to what matters."
  3. "Wake up to something extraordinary."


## 14. Pattern 13 — Prompt Chaining (Multi-Step Reasoning Pipeline)

In [14]:
review = "The delivery was 3 days late and the box was damaged, but the product itself works great and customer support resolved my refund quickly."

# Step 1: extract structured facts
step1 = ask(f"Extract the issues and positives from this review as two short bullet lists:\n\n{review}")
print("STEP 1 — Extraction:\n", step1)

# Step 2: feed step1's output into a second prompt
step2 = ask(f"Based on this analysis:\n{step1}\n\nWrite a one-sentence customer support response acknowledging the issues and thanking them for the positive feedback.")
print("\nSTEP 2 — Response generation:\n", step2)

STEP 1 — Extraction:
 # Issues
- Delivery was 3 days late
- Box arrived damaged

# Positives
- Product works great
- Customer support resolved refund quickly

STEP 2 — Response generation:
 # Customer Support Response

Thank you for your feedback—we sincerely apologize for the late delivery and damaged packaging, but we're glad the product works well and that our team could quickly resolve the refund for you.


## 15. Pattern 14 — Self-Consistency (Sample Multiple Times, Pick Majority)

In [15]:
import collections

question = "A bat and ball cost $1.10 together. The bat costs $1 more than the ball. How much does the ball cost?"

answers = []
for i in range(5):
    response = client.messages.create(model=MODEL, max_tokens=200, temperature=0.8,
                                       messages=[{"role": "user", "content": question + " Answer with just the final number in dollars."}])
    answers.append(response.content[0].text.strip())

print("All 5 answers:", answers)
print("\nMost common answer:", collections.Counter(answers).most_common(1))

All 5 answers: ['0.05', '0.05', '0.05', '0.05', '0.05']

Most common answer: [('0.05', 5)]


## 16. Summary — Patterns Covered Today

| # | Pattern | Key Lesson |
|---|---------|------------|
| 1 | Zero-shot | No examples — fast but verbose/inconsistent output |
| 2 | Few-shot | 2 examples made output format dramatically more consistent |
| 3 | Chain-of-thought | Step-by-step reasoning improved multi-step math accuracy |
| 4 | System prompts | Controlled persona, tone, and length persistently |
| 5 | Structured JSON | Got machine-parseable output for downstream code |
| 6 | JSON parsing fix | Models add markdown fences even when told not to — strip defensively |
| 7 | Tool use | Model requested external data via structured function call |
| 8 | Role/persona | Same question, different audience-appropriate answers |
| 9 | Output format constraints | Numbered list, word limits — kills unsolicited extra text |
| 10 | XML tags | Cleanly separated instructions, data, and format rules |
| 11 | Negative examples | "Don't" instructions are as effective as "do" instructions |
| 12 | Temperature | 0.0 = consistent, 1.0 = varied — pick based on the task |
| 13 | Prompt chaining | Breaking a task into steps beats one giant prompt |
| 14 | Self-consistency | Multiple samples + majority vote catches model inconsistency |

**Biggest takeaway:** the model's raw capability matters less than how you structure
the prompt. The same Haiku model went from a hedgy paragraph (zero-shot) to a clean
one-word answer (few-shot) just by changing the prompt — not the model.